# Level 2 — Task 2: Wine Quality Prediction

## Objective
Build a machine-learning model to predict wine quality from physicochemical properties.

## Dataset
The notebook uses the public **UCI Wine Quality — Red Wine** dataset. It contains physicochemical measurements such as acidity, sugar, chlorides, sulfur dioxide, density, pH, sulphates and alcohol, with a `quality` score from 3 to 8.

The notebook downloads the dataset automatically if it is not already present in the `data` folder.

## Workflow
1. Load and inspect the wine-quality data.
2. Explore the quality distribution and numerical features.
3. Check missing values and duplicates.
4. Convert the quality score into a binary target:
   - **Good wine:** quality >= 7
   - **Not good:** quality < 7
5. Standardize features.
6. Train Logistic Regression.
7. Evaluate using accuracy, precision, recall and F1-score.
8. Visualize the confusion matrix.
9. Compare a Random Forest model as an additional baseline.
10. Inspect feature importance.
11. Generate practical conclusions.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn.


In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
DATA_PATH = "../data/winequality-red.csv"

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH, sep=";")

print("Shape:", df.shape)
display(df.head())


## 1. Initial inspection


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nQuality distribution:")
display(df["quality"].value_counts().sort_index())


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="quality")
plt.title("Wine Quality Distribution")
plt.xlabel("Quality Score")
plt.ylabel("Number of Wines")
plt.show()


**Observation:** The quality score is not evenly distributed. Most observations are concentrated around the middle quality scores, while very high-quality wines are less common. This makes class imbalance an important consideration when converting quality into a binary prediction problem.


## 2. Explore physicochemical features


In [ ]:
display(df.describe().T)

plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()


## 3. Data cleaning

The dataset is checked for missing values and exact duplicate rows. Duplicate rows are removed before modelling because repeated identical records can distort model training.


In [ ]:
print("Missing values before cleaning:", int(df.isna().sum().sum()))
duplicates = int(df.duplicated().sum())

df = df.drop_duplicates().copy()

print("Duplicates removed:", duplicates)
print("Shape after duplicate removal:", df.shape)


## 4. Create the prediction target

For a practical binary classification problem:
- `1` = Good wine (`quality >= 7`)
- `0` = Not good (`quality < 7`)

This target is intentionally defined in the notebook so the classification objective is clear.


In [ ]:
df["good_wine"] = (df["quality"] >= 7).astype(int)

print("Target distribution:")
display(df["good_wine"].value_counts())
display(df["good_wine"].value_counts(normalize=True).rename("proportion"))


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="good_wine")
plt.title("Good vs Not-Good Wine")
plt.xlabel("Target: 1 = Good (quality >= 7)")
plt.ylabel("Number of Wines")
plt.show()


## 5. Train-test split

The original `quality` score is excluded from the input features because it is the source used to create the target. Keeping it would cause target leakage.

The remaining physicochemical measurements are used as predictors.


In [ ]:
target = "good_wine"
drop_cols = ["quality", "good_wine"]

X = df.drop(columns=drop_cols)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 6. Logistic Regression

A pipeline is used so scaling is fitted only on the training data. This prevents test-set information from leaking into preprocessing.


In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

logistic_model.fit(X_train, y_train)
log_pred = logistic_model.predict(X_test)

log_metrics = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy_score(y_test, log_pred),
    "Precision": precision_score(y_test, log_pred, zero_division=0),
    "Recall": recall_score(y_test, log_pred, zero_division=0),
    "F1": f1_score(y_test, log_pred, zero_division=0)
}

print(classification_report(y_test, log_pred, target_names=["Not Good", "Good"], zero_division=0))
pd.DataFrame([log_metrics])


## 7. Confusion matrix


In [ ]:
cm = confusion_matrix(y_test, log_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Not Good", "Good"],
    yticklabels=["Not Good", "Good"]
)
plt.title("Logistic Regression — Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


**Observation:** The confusion matrix separates false positives and false negatives. For quality screening, recall for the `Good` class can be especially useful when the business wants to identify as many potentially good wines as possible.


## 8. Random Forest comparison

A Random Forest is added as a nonlinear comparison model. This helps determine whether the relationships between physicochemical properties and wine quality are better captured by a tree-based model.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_metrics = {
    "Model": "Random Forest",
    "Accuracy": accuracy_score(y_test, rf_pred),
    "Precision": precision_score(y_test, rf_pred, zero_division=0),
    "Recall": recall_score(y_test, rf_pred, zero_division=0),
    "F1": f1_score(y_test, rf_pred, zero_division=0)
}

print(classification_report(y_test, rf_pred, target_names=["Not Good", "Good"], zero_division=0))


In [ ]:
results = pd.DataFrame([log_metrics, rf_metrics])
display(results)

results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1"]].plot(
    kind="bar", figsize=(10, 6)
)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Model Performance Comparison")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=.25)
plt.show()


## 9. Random Forest feature importance


In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

plt.figure(figsize=(10, 7))
sns.barplot(data=importance, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("")
plt.show()


## 10. Final conclusions

- Wine quality can be modelled from physicochemical properties as a classification problem.
- The target definition (`quality >= 7`) makes the task directly interpretable as identifying higher-quality wines.
- Logistic Regression provides an interpretable linear baseline.
- Random Forest provides a nonlinear comparison and can capture interactions between chemical properties.
- Feature importance helps identify which measurements contribute most to the tree-based model.

### Business use
A winery could use a model like this as a screening tool to identify batches with characteristics associated with higher quality before further expert testing.

### Limitations
- Quality labels are based on the original dataset's expert score and may contain subjectivity.
- The threshold of 7 is a project definition, not a universal industry standard.
- The model should be validated on new winery-specific data before operational use.


In [ ]:
# Save outputs
results.to_csv("../output/model_comparison.csv", index=False)
importance.to_csv("../output/feature_importance.csv", index=False)

prediction_output = pd.DataFrame({
    "Actual": y_test.values,
    "Logistic_Predicted": log_pred,
    "RandomForest_Predicted": rf_pred
})
prediction_output.to_csv("../output/test_predictions.csv", index=False)

print("Wine Quality Prediction task completed.")
